## ver 1.2
> cutmix 사용안함
> 
> seed는 고정
> 
> classifier학습 - 전체학습 2단계 (전체 학습시는 lr을 1e-5로 함. 1e-4는 과적합이 빨리 올수 있다는 충고반영)
> 
> BatchNorm freeze는 안함
>
> Grad-CAM 적용
>
> 오답클래스 데이터프레임화. 오답클래스 GradCAM visualize 필요
> 


In [ ]:
import pandas as pd
import numpy as np
import os
from pathlib import Path
import cv2
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from albumentations import ToTensorV2
import albumentations as A
import random

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset,DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.optim import Adam
from torchmetrics.classification import MulticlassF1Score, MulticlassAccuracy,MulticlassRecall

In [ ]:
def seed_everything(seed: int = 42):
    random.seed(seed)          # python random
    np.random.seed(seed)       # numpy
    torch.manual_seed(seed)    # torch CPU
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)

base_path=Path(r"/kaggle/input/datasets/alxmamaev/flowers-recognition/flowers")
path_list=[]

for img_path in base_path.rglob("*.jpg"):
    path_list.append({"label":img_path.parent.name,"path":img_path})
   
df=pd.DataFrame(path_list)
df['targets']=pd.factorize(df['label'])[0]
df=df.sample(frac=1,random_state=42).reset_index(drop=True)
train_df,tmp_df=train_test_split(df,test_size=0.3,stratify=df['label'],random_state=42)
val_df,test_df=train_test_split(tmp_df,test_size=0.4,stratify=tmp_df['label'],random_state=42)

In [ ]:
img_aug=A.Compose([
    A.RandomResizedCrop(size=(224,224),scale=(0.8,1.0),ratio=(0.9,1.1),p=1),
    A.HorizontalFlip(p=0.3),
    A.Affine(scale=(0.9,1.1),rotate=(-15,15),border_mode=cv2.BORDER_REFLECT_101,p=0.3),
    A.ColorJitter(brightness=0.2,contrast=0.2,saturation=0.2,hue=0.03,p=0.6),
    A.CoarseDropout(num_holes_range=(1, 1), hole_height_range=(48, 48),
                    hole_width_range=(48, 48),fill=0,p=0.25)
])

tr_resnet34=A.Compose([
    A.RandomResizedCrop(size=(224,224),scale=(0.8,1.0),ratio=(0.9,1.1),p=1),
    A.HorizontalFlip(p=0.3),
    A.Affine(scale=(0.9,1.1),rotate=(-15,15),border_mode=cv2.BORDER_REFLECT_101,p=0.3),
    A.ColorJitter(brightness=0.2,contrast=0.2,saturation=0.2,hue=0.03,p=0.6),
    A.CoarseDropout(num_holes_range=(1, 1), hole_height_range=(48, 48),
                    hole_width_range=(48, 48),fill=0,p=0.25),
    A.Normalize(mean=(0.485, 0.456, 0.406),std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

val_resnet34=A.Compose([
    A.Resize(224,224,p=1),
    A.Normalize(mean=(0.485, 0.456, 0.406),std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

In [ ]:
def visualize(df,nrows=4,ncols=4,augment=None):
    df=df.sample(min(nrows*ncols,len(df)))
    fig,axs=plt.subplots(nrows,ncols,figsize=(ncols*3,nrows*3))
    axs=axs.flatten()
    for ax,(_,row) in zip(axs,df.iterrows()):
        img=cv2.imread(row['path'])
        img=cv2.cvtColor(img,cv2.COLOR_BGR2RGB)
        if augment is not None:
            img=augment(image=img)['image']
        H,W=img.shape[:2]
        label=row['label']
        ax.imshow(img)
        ax.set_title(f"{label}\n{H}x{W}")
    plt.show()

visualize(df,augment=img_aug)

In [ ]:
class FlowerCustom(Dataset):
    def __init__(self,path,targets,augment=None):
        self.path=path
        self.targets=targets
        self.augment=augment
    def __len__(self):
        return len(self.path)
    def __getitem__(self,idx):
        img=cv2.imread(self.path[idx])
        img=cv2.cvtColor(img,cv2.COLOR_BGR2RGB)
        if self.augment is None:
            raise ValueError(f"IMG Augment must be need")
        img=self.augment(image=img)['image']
        targets=torch.tensor(self.targets[idx],dtype=torch.long)
        return img,targets,str(self.path[idx])    # self.path[idx] (현재 샘플의 경로)만 반환하는 게 맞다. self.path[idx]는 PosixPath객체라 그냥은 못 묶어줌. str으로 변환


train_custom=FlowerCustom(train_df['path'].to_list(),train_df['targets'].to_list(),
                          augment=tr_resnet34)
val_custom=FlowerCustom(val_df['path'].to_list(),val_df['targets'].to_list(),
                        augment=val_resnet34)
test_custom=FlowerCustom(test_df['path'].to_list(),test_df['targets'].to_list(),
                         augment=val_resnet34)

train_loader=DataLoader(train_custom,batch_size=32,shuffle=True,num_workers=4,pin_memory=True)
val_loader=DataLoader(val_custom,batch_size=32,shuffle=False,num_workers=4,pin_memory=True)
test_loader=DataLoader(test_custom,batch_size=32,shuffle=False,num_workers=4,pin_memory=True)

In [ ]:
from torchvision.models import efficientnet_b1,EfficientNet_B1_Weights

device="cuda" if torch.cuda.is_available() else "cpu"

weights = EfficientNet_B1_Weights.IMAGENET1K_V2
model=efficientnet_b1(weights=weights).to(device)
model.classifier[1]=nn.Linear(in_features=1280, out_features=5, bias=True).to(device)

for p in model.parameters():
    p.requires_grad=False
for p in model.classifier.parameters():
    p.requires_grad=True
optimizer=Adam(model.classifier.parameters(),lr=1e-3,weight_decay=1e-4)
scheduler=ReduceLROnPlateau(optimizer,factor=0.1,patience=3)
loss_func=nn.CrossEntropyLoss()
metric_rec=MulticlassRecall(num_classes=5,average="macro").to(device)
metric_acc=MulticlassAccuracy(num_classes=5).to(device)
metric_f1=MulticlassF1Score(num_classes=5,average='macro').to(device)



In [ ]:
from torchinfo import summary
summary(model,input_size=(16,3,224,224),col_names=["input_size", "output_size", "num_params"])

for i, layer in enumerate(model.modules()):
    print(i, layer.__class__.__name__)

In [ ]:
from typing import List
from dataclasses import dataclass,field
from tqdm import tqdm

@dataclass
class History:
    training_accuracy:List[float]=field(default_factory=list)
    training_recall:List[float]=field(default_factory=list)
    training_loss:List[float]=field(default_factory=list)
    val_accuracy:List[float]=field(default_factory=list)
    val_recall:List[float]=field(default_factory=list)
    val_loss:List[float]=field(default_factory=list)
history=History()


class Trainer:
    def __init__(self,train_loader,val_loader,model,optimizer,loss_func,
                 scheduler,metric_acc,metric_rec,device,history,mode="min"):
        self.model=model
        self.train_loader=train_loader
        self.val_loader=val_loader
        self.optimizer=optimizer
        self.loss_func=loss_func
        self.scheduler=scheduler
        self.metric_acc=metric_acc
        self.metric_rec=metric_rec
        self.device=device
        self.history=history
        if mode=="max":
            self.best_value=float('-inf')
        else:
            self.best_value=float('inf')

    def training_epoch(self,epoch):
        self.metric_acc.reset()
        self.metric_rec.reset()
        self.model.train()
        loss_sum=0.0
        avg_loss=0.0
        with tqdm(total=len(self.train_loader),desc=f"training {epoch}",leave=True) as bar:
            for batch_idx,(x_train,y_train,_) in enumerate(self.train_loader):
                x_train=x_train.to(self.device)
                y_train=y_train.to(self.device)
                logits=self.model(x_train)
                loss=self.loss_func(logits,y_train)
                self.optimizer.zero_grad()
                loss.backward()
                self.optimizer.step()
                loss_sum+=loss.item()
                avg_loss=loss_sum/(batch_idx+1)
                preds=logits.argmax(dim=1)   # dim=-1과 같다. (B,12)  1번 dim 즉 행에 대해서
                self.metric_acc.update(preds, y_train)
                self.metric_rec.update(preds, y_train)
                bar.update(1)

                if batch_idx%10==0:
                    acc=self.metric_acc.compute().item()
                    recall=self.metric_rec.compute().item()
                    bar.set_postfix({"acc": acc, "recall":recall, "loss":avg_loss,"epoch":epoch})
            return self.metric_acc.compute().item(), self.metric_rec.compute().item(),avg_loss  

    def validating_epoch(self,epoch):
        self.metric_acc.reset()
        self.metric_rec.reset()
        self.model.eval()
        loss_sum=0
        avg_loss=0.0
        with tqdm(total=len(self.val_loader),desc=f"validating {epoch}", leave=True) as bar:
            with torch.no_grad():
                for batch_idx,(x_val,y_val,_) in enumerate(self.val_loader):
                    x_val=x_val.to(self.device)
                    y_val=y_val.to(self.device)
                    logits=self.model(x_val)
                    loss=self.loss_func(logits,y_val)

                    preds=logits.argmax(dim=-1)
                    self.metric_acc.update(preds,y_val)
                    self.metric_rec.update(preds,y_val)
                    loss_sum+=loss.item()
                    avg_loss=loss_sum/(batch_idx+1)
                    bar.update(1)
                    if batch_idx%10==0:
                        acc=self.metric_acc.compute().item()
                        recall=self.metric_rec.compute().item()
                        bar.set_postfix({"acc": acc, "recall":recall, "loss":avg_loss,"epoch":epoch})
                return self.metric_acc.compute().item(), self.metric_rec.compute().item(),avg_loss

    
    def fit(self,epochs,early_stop,path):
        stop_count=0   
        for epoch in range(epochs):
            training_accuracy,training_recall,training_loss=self.training_epoch(epoch)
            self.history.training_accuracy.append(training_accuracy)
            self.history.training_recall.append(training_recall)
            self.history.training_loss.append(training_loss)
            val_accuracy,val_recall,val_loss=self.validating_epoch(epoch)
            self.history.val_accuracy.append(val_accuracy)
            self.history.val_recall.append(val_recall)
            self.history.val_loss.append(val_loss)
            self.scheduler.step(val_loss)   # scheduler는 early_stop >= scheduler.patience + 1정도가 안정적. ex)scheduler patience = 3이면 early_stop = 5
            
            if self.best_value>val_loss:
                self.best_value=val_loss
                stop_count=0
                torch.save(self.model.state_dict(),os.path.join(path,f"{epoch}_{val_loss}.pt"))
            else:
                stop_count+=1
                if stop_count>=early_stop:
                    print(f"early_stopped. current epoch : {epoch}")
                    return self.history
                    
        return self.history



In [ ]:
output_path=r"/kaggle/working/"

t=Trainer(train_loader,val_loader,model,optimizer,loss_func,scheduler,metric_acc,metric_rec,device,history,mode="min")
history=t.fit(5,2,output_path)

In [ ]:

best_param=torch.load(r"/kaggle/working/4_0.3316377234458923.pt")
model.load_state_dict(best_param)
for p in model.parameters():
    p.requires_grad=True
optimizer=Adam(model.parameters(),lr=1e-5,weight_decay=1e-4)    # 1e-4는 3,000장에선 다소 공격적
t1=Trainer(train_loader,val_loader,model,optimizer,loss_func,scheduler,metric_acc,metric_rec,device,history,mode="min")
history=t1.fit(15,5,output_path)

In [ ]:
final_param=torch.load(r"/kaggle/working/14_0.17188705176115035.pt")
model.load_state_dict(final_param)


class Predict:
    def __init__(self, model, test_loader, device):
        self.model = model
        self.test_loader = test_loader
        self.actual_list = []
        self.pred_list = []
        self.wrong_info = []
        self.device = device

    def predict(self):
        self.model.eval()

        with tqdm(total=len(self.test_loader), desc="predicting", leave=True) as bar:
            for x_test, y_test, paths in self.test_loader:
                x_test = x_test.to(self.device)
                y_test = y_test.to(self.device)

                self.actual_list.extend(y_test.detach().cpu().numpy())

                with torch.no_grad():
                    logits = self.model(x_test)
                    probs = F.softmax(logits, dim=1)
                    preds = torch.argmax(logits, dim=1)
                    self.pred_list.extend(preds.detach().cpu().numpy())
                    confs = probs.max(dim=1).values

                    wrong_mask = preds != y_test             # "배치 크기만큼의 bool 텐서"가 만들어진다. ex) tensor([False,True,False..]) element-wise비교
                    wrong_paths = [                          # preds!=y_test인 경우 True
                        paths[i] for i in range(len(paths))
                        if wrong_mask[i].item()              # item() => tensor boolean을 python boolean으로 바꿔서 if문 구동하기 위해 
                    ]
                    wrong_labels = y_test[wrong_mask].cpu()  # pytorch는 boolean indexing이 된다.  그냥 python에서는 불가.          
                    wrong_preds = preds[wrong_mask].cpu()
                    wrong_confs = confs[wrong_mask].cpu()

                    # 4. 저장
                    for path, true_label, pred_label, conf in zip(
                        wrong_paths, wrong_labels, wrong_preds, wrong_confs
                    ):
                        self.wrong_info.append({
                            "path": path,
                            "true_label": true_label.item(),
                            "pred_label": pred_label.item(),
                            "confidence": round(conf.item(), 4)
                        })

                bar.update(1)

        return self.actual_list, self.pred_list, self.wrong_info


In [ ]:
p=Predict(model,test_loader,device)
actual_list,pred_list,wrong_info=p.predict()

In [ ]:
from sklearn.metrics import confusion_matrix


cm=confusion_matrix(actual_list,pred_list)
print(cm)

wrong_df=pd.DataFrame(wrong_info)
wrong_df

In [ ]:
# target_layer = model.features[-1][0]

class GradCAM_V2:
    def __init__(self,model,idx1,idx2):
        self.model=model
        self.target_layer=self.model.features[idx1][idx2]
        self.feature_map=None
        self.gradient=None
        self.fwd_handle=self.target_layer.register_forward_hook(self._forward)
        self.bwd_handle=self.target_layer.register_full_backward_hook(self._backward)
    def remove_hook(self):
        self.fwd_handle.remove()
        self.bwd_handle.remove()
    def _forward(self,module,inputs,outputs):
        self.feature_map=outputs
    def _backward(self,module,grad_inputs,grad_outputs):
        self.gradient=grad_outputs[0]
    @torch.no_grad()
    def _normalize(self,cam):                                    # 입력된 cam shape (16,1,224,224)
        batch_size=cam.shape[0]                                     
        cam_flatten=cam.view(batch_size,-1)                      # (16,1*224*224) => (16, 50176)     
        cam_min=cam_flatten.min(dim=1)[0].view(batch_size,1,1,1) # (16,).view(batch_size,1,1,1) => (16,1,1,1)    . min(dim=1) 반환값 (value, indices) 그래서 [0] indexing
        cam_max=cam_flatten.max(dim=1)[0].view(batch_size,1,1,1) # (16,).view(batch_size,1,1,1) => (16,1,1,1)
        cam=(cam-cam_min)/(cam_max-cam_min+1e-8)                 # broadcasting => (16, 1, 224, 224)
        return cam                                               # batch 16장의 Grad-CAM
    def generate(self,x,class_idx=None):    # x가 (16,3,224,224).  num_classes는 5로 가정. 
        self.model.eval()
        self.model.zero_grad(set_to_none=True)
        logits=self.model(x)                      # (16, 5)
        preds=logits.argmax(dim=-1)               # (16,)
        batch_size=logits.shape[0]                # (16,)
        if class_idx is None:
            target_idx=preds                       # (16,)
        elif isinstance(class_idx,int):
            target_idx=torch.full((batch_size,),class_idx,device=logits.device,dtype=torch.long) #16
            # torch.full(size,value) => torch.full(4,3) : [3,3,3,3]
            # torch.full의 인자인 size는 꼭 튜플값일 필요는 없지만 단일정수 불가. (16,) , (4,3), 또는 리스트 이런식
        else:
            target_idx=class_idx.to(logits.device)
        score_each=logits.gather(1,target_idx.unsqueeze(1)).squeeze(1)  # (16,)
        # gather(dim,index)는 dim 방향으로, index가 가리키는 위치의 값들을 뽑아라
        # target_idx.unsqueeze(1) : (16,1)
        # logits.gather(dim, index) : logits <= (16,5)  index(target_idx.unsqueeze(1)) <= (16,1)
        # score_each <= (16,1) 이것을 squeeze(1)하니깐 최종적으로 (16,)

        score=score_each.sum()  # backward는 scalar값에 대해서 호출 가능하기에   <== 0차원 텐서 스칼라값이 된다. ex)torch.tensor([1,2,3])은 tensor(6) 스칼라
                                # score를 더했으니 CAM도 하나로 합쳐지는 것 아니다!!! autograd가 각 샘플별 계산 그래프를 따로 가지고 있기 때문에 .. 추후 숙지필요 
        score.backward()
        grad=self.gradient                         # 후킹한곳의 shape는 (16,256,56,56)
        if grad is not None:
            print(grad.shape)
        else:
            print("type None")
        act=self.feature_map                       # (16,256,56,56)   
        weights=grad.mean(dim=(2,3),keepdim=True)  # (16,256,1,1)
        cam=(act*weights).sum(dim=1,keepdim=True)  # (16,1,56,56) <= 브로드캐스팅 되고 dim 1을 sum후 keep
        cam=F.relu(cam)
        cam=F.interpolate(cam,size=(x.shape[2],x.shape[3]),mode='bilinear',align_corners=False) # (16,1,224,224)
        cam=cam.detach()
        cam=self._normalize(cam) #_normalize 호출. 입력 shape는 (16,1,224,224) 반환값도 => (16, 1, 224, 224) 
        return cam.cpu()

print("done2")

In [ ]:
class WrongPredict(Dataset):
    def __init__(self,path,true_label,augment=val_resnet34):
        super().__init__()
        self.path=path
        self.true_label=true_label
        self.augment=augment
    def __len__(self):
        return len(self.path)
    def __getitem__(self,idx):
        img=cv2.cvtColor(cv2.imread(str(self.path[idx])),cv2.COLOR_BGR2RGB)
        img=self.augment(image=img)['image']
        true_label=torch.tensor(self.true_label[idx],dtype=torch.long)
        return img,true_label

wrong_custom=WrongPredict(wrong_df['path'].to_list(),
                          wrong_df['true_label'].to_list())

wrong_loader=DataLoader(wrong_custom,batch_size=len(wrong_custom),shuffle=False,num_workers=4,pin_memory=True)
print("done2")

In [ ]:
def visualizer(dataloader, df, nrows=5):
    IMG, Y_TRUE = next(iter(dataloader))
    IMG = IMG.to(device)

    num_image = min(IMG.shape[0], nrows)

    gradcam = GradCAM_V2(model, idx1=-1, idx2=0)
    cam = gradcam.generate(IMG)
    gradcam.remove_hook()

    print("IMG min/max:", IMG.min().item(), IMG.max().item())
    print("CAM min/max:", cam.min().item(), cam.max().item())

    images = IMG.detach().cpu()
    cam = cam.detach().cpu()

    mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 1, 3)  # img는 permute(1,2,0)을 해서 (H,W,3) 브로드캐스팅을 위해 (1,1,3)변환
    std = torch.tensor([0.229, 0.224, 0.225]).view(1, 1, 3)

    fig, axes = plt.subplots(nrows=nrows, ncols=3, figsize=(9, 3 * nrows))

    if nrows == 1:
        axes = axes.reshape(1, 3)

    for i in range(num_image):
        img = images[i].permute(1, 2, 0)
        img = (img * std + mean).clamp(0, 1)   # 역정규화. clam(0,1) => 값을 0~1사이로 제한

        cmp = cam[i].squeeze(0)                # (H, W)

        true_label = df['true_label'].iloc[i]
        pred = df['pred_label'].iloc[i]
        confidence = df['confidence'].iloc[i]

        axes[i, 0].imshow(img)
        axes[i, 0].set_title(f"Original\nConf: {confidence:.3f}\nTrue: {true_label}, Pred: {pred}")
        axes[i, 0].axis("off")

        axes[i, 1].imshow(cmp, cmap="jet")
        axes[i, 1].set_title("Grad-CAM")
        axes[i, 1].axis("off")

        axes[i, 2].imshow(img)
        axes[i, 2].imshow(cmp, cmap="jet", alpha=0.45)
        axes[i, 2].set_title("Overlay")
        axes[i, 2].axis("off")

    for i in range(num_image, nrows):
        for j in range(3):
            axes[i, j].axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
visualizer(wrong_loader,wrong_df)